# 1.2 Train preprocessing

Fit a preprocessor on **train only**. The pipeline itself drops duplicates, drops rows with missing `price`, then imputes the remaining columns (no new features).

| Artifact | Path |
|---|---|
| Input | `1-experimentation/data/data_raw_train.csv` |
| Output CSV | `1-experimentation/data/data_train_preprocessed.csv` |
| Output preprocessor | `1-experimentation/models/preprocessor.pkl` |


In [1]:
%pip install -q pandas scikit-learn



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pickle
from pathlib import Path

import pandas as pd
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

set_config(transform_output="pandas")

TARGET = "price"
FEATURE_NUMERIC = ["sqft", "bedrooms", "bathrooms", "year_built"]
FEATURE_CATEGORICAL = ["location", "condition"]

TRAIN_PATH = "../data/data_raw_train.csv"
PREPROCESSED_TRAIN_PATH = "../data/data_preprocessed_train.csv"
PREPROCESSOR_PATH = "../models/preprocessor.pkl"

Path("../models").mkdir(parents=True, exist_ok=True)


## 1. Load train

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
print(f"Rows: {train_df.shape[0]} | duplicates={train_df.duplicated().sum()} | missing price={int(train_df[TARGET].isna().sum())}")
train_df.head()


Rows: 68 | duplicates=0 | missing price=1


,price,sqft,bedrooms,bathrooms,location,year_built,condition
0,495000.0,1950,3,2.0,Urban,1981,Good
1,320000.0,1700,2,1.5,Rural,1961,Fair
2,615000.0,2230,3,2.0,Downtown,1986,Good
3,398000.0,1680,2,2.0,Suburb,1968,Fair
4,357000.0,1580,2,1.5,Suburb,1960,Fair


## 2. Fit preprocessor and save artifacts

The pickled pipeline includes:

1. Remove duplicate rows
2. Drop rows with missing `price`
3. Impute numeric columns with the median and categorical columns with the most frequent value


In [4]:
class DropDuplicates(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop_duplicates().reset_index(drop=True)


class DropMissingPrice(BaseEstimator, TransformerMixin):
    def __init__(self, target="price"):
        self.target = target

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.dropna(subset=[self.target]).reset_index(drop=True)


preprocessor = Pipeline(
    [
        ("drop_duplicates", DropDuplicates()),
        ("drop_missing_price", DropMissingPrice(target=TARGET)),
        (
            "impute",
            ColumnTransformer(
                transformers=[
                    ("num", SimpleImputer(strategy="median"), FEATURE_NUMERIC),
                    ("cat", SimpleImputer(strategy="most_frequent"), FEATURE_CATEGORICAL),
                ],
                remainder="passthrough",
                verbose_feature_names_out=False,
            ),
        ),
    ]
)

preprocessor.fit(train_df)
train_preprocessed = preprocessor.transform(train_df)[train_df.columns]

print(
    f"After preprocessor: {train_preprocessed.shape[0]} rows, "
    f"duplicates={train_preprocessed.duplicated().sum()}, "
    f"missing price={int(train_preprocessed[TARGET].isna().sum())}"
)

train_preprocessed.to_csv(PREPROCESSED_TRAIN_PATH, index=False)
with open(PREPROCESSOR_PATH, "wb") as file:
    pickle.dump(preprocessor, file)

print(f"Wrote {PREPROCESSED_TRAIN_PATH}")
print(f"Wrote {PREPROCESSOR_PATH}")


After preprocessor: 67 rows, duplicates=0, missing price=0
Wrote ../data/data_preprocessed_train.csv
Wrote ../models/preprocessor.pkl
